# Multiplier Regression Training (Clean, NumPy-only)

This notebook has no sklearn/pandas imports.
It trains a linear matrix model and writes:
- `backend/regression_layer/multiplier_model_matrix.json`
- `backend/regression_layer/metrics.json`


In [1]:
import json
from pathlib import Path
import numpy as np


In [2]:
BASE_DIR = Path.cwd().resolve().parent.parent
NUTRITION_PATH = BASE_DIR / "backend" / "nutrition.json"
REGRESSION_DIR = BASE_DIR / "backend" / "regression_layer"
MODEL_OUT_PATH = REGRESSION_DIR / "multiplier_model_matrix.json"
METRICS_OUT_PATH = REGRESSION_DIR / "metrics.json"
NUTRIENT_KEYS = ("calories", "carbs_g", "protein_g", "fats_g", "sugar_g", "salt_mg")

REGRESSION_DIR.mkdir(parents=True, exist_ok=True)
print("Model output:", MODEL_OUT_PATH)
print("Metrics output:", METRICS_OUT_PATH)


Model output: C:\Users\Asus\LocalFile\Desktop\RasaRight\backend\regression_layer\multiplier_model_matrix.json
Metrics output: C:\Users\Asus\LocalFile\Desktop\RasaRight\backend\regression_layer\metrics.json


In [3]:
def clamp01(v: float) -> float:
    return max(0.0, min(1.0, float(v)))


def target_multipliers(size: float, sweetness: float, saltiness: float, oiliness: float) -> np.ndarray:
    s = clamp01(size)
    sw = clamp01(sweetness)
    sa = clamp01(saltiness)
    oi = clamp01(oiliness)
    vals = np.array([
        0.75 + 0.90 * s + 0.20 * oi + 0.05 * sw,
        0.80 + 0.70 * s + 0.25 * sw,
        0.85 + 0.50 * s,
        0.75 + 0.60 * s + 0.55 * oi,
        0.75 + 0.35 * s + 0.55 * sw,
        0.75 + 0.35 * s + 0.70 * sa,
    ], dtype=float)
    return np.clip(vals, 0.1, 3.0)


def load_food_count() -> int:
    with NUTRITION_PATH.open("r", encoding="utf-8") as f:
        nutrition = json.load(f)
    return max(1, len(nutrition))


def make_dataset(samples_per_food: int = 300, noise_std: float = 0.03):
    food_count = load_food_count()
    total_samples = food_count * samples_per_food
    X = np.random.rand(total_samples, 4).astype(float)
    y = np.zeros((total_samples, len(NUTRIENT_KEYS)), dtype=float)
    for i in range(total_samples):
        s, sw, sa, oi = X[i]
        t = target_multipliers(s, sw, sa, oi)
        noise = np.random.normal(0.0, noise_std, size=t.shape)
        y[i] = np.clip(t + noise, 0.1, 3.0)
    return X, y


In [4]:
X, y = make_dataset(samples_per_food=300)

rng = np.random.default_rng(42)
idx = np.arange(len(X))
rng.shuffle(idx)
split = int(len(idx) * 0.8)
train_idx, test_idx = idx[:split], idx[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

X_train_aug = np.concatenate([X_train, np.ones((len(X_train), 1))], axis=1)
coef_aug, _, _, _ = np.linalg.lstsq(X_train_aug, y_train, rcond=None)

weights = coef_aug[:4, :]
bias = coef_aug[4, :]

preds = np.clip(X_test @ weights + bias, 0.1, 3.0)
mae = float(np.mean(np.abs(y_test - preds)))
ss_res = float(np.sum((y_test - preds) ** 2))
ss_tot = float(np.sum((y_test - np.mean(y_test, axis=0)) ** 2))
r2 = float(1.0 - (ss_res / ss_tot)) if ss_tot > 0 else 0.0

model_payload = {
    "model_type": "linear_matrix",
    "features": ["size", "sweetness", "saltiness", "oiliness"],
    "outputs": list(NUTRIENT_KEYS),
    "weights": weights.tolist(),
    "bias": bias.tolist(),
}
with MODEL_OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(model_payload, f, indent=2)

metrics = {
    "model_type": "linear_matrix",
    "samples": int(len(X)),
    "test_size": 0.2,
    "mae": mae,
    "r2": r2,
    "features": ["size", "sweetness", "saltiness", "oiliness"],
    "outputs": list(NUTRIENT_KEYS),
    "model_path": str(MODEL_OUT_PATH)
}
with METRICS_OUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved model:", MODEL_OUT_PATH)
print("Saved metrics:", METRICS_OUT_PATH)
print("MAE:", round(mae, 4), "R2:", round(r2, 4))


Saved model: C:\Users\Asus\LocalFile\Desktop\RasaRight\backend\regression_layer\multiplier_model_matrix.json
Saved metrics: C:\Users\Asus\LocalFile\Desktop\RasaRight\backend\regression_layer\metrics.json
MAE: 0.0241 R2: 0.9809
